In [ ]:
# read pickle file
import pandas as pd

df8918 = pd.read_pickle('****.pkl')
df8918

In [ ]:
df8918['event_clinical_yesno'].value_counts()

In [ ]:
df8918 = df8918[df8918['event_clinical_yesno'].isin([0, 1])]
df8918

In [ ]:
df8918['sid'].value_counts()

In [ ]:
# read xlxs file
import pandas as pd
meta = pd.read_excel('*******.xlsx')
meta

In [ ]:
meta_for_merge = meta.rename(columns={'zanaid': 'sid'})[['sid', 'Gender']]
df8918 = df8918.merge(meta_for_merge, on='sid', how='left')

In [ ]:
df8918['Gender'].value_counts()

In [ ]:
# Freevc: Towards High-Quality Text-Free One-Shot Voice Conversion，ICASSP 2023 - 2023 IEEE Inter

In [ ]:
import os
import re
from pathlib import Path
from TTS.api import TTS
import torch
import pandas as pd
df8918 = pd.read_pickle('*****.pkl')
df8918 = df8918[df8918['event_clinical_yesno'].isin([0, 1])]
df8918

In [ ]:
# remove the target 1956 from df8918
df8918 = df8918[df8918['sid'] != 1956]
df8918

In [ ]:
tacticas_root = Path("*****")   
output_root = Path("*****")  #Save path for converted files

# target wav
target_wav = "*****.wav"  # target speech path

In [ ]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

print("Loading voice conversion model...")
tts = TTS("voice_conversion_models/multilingual/vctk/freevc24").to(device)
print("Model loaded.")


In [ ]:

sid_to_files = df8918.groupby('sid')['filename'].apply(set).to_dict()
sid_to_files

In [ ]:
# Traverse the tacticas folder
uid_pattern = re.compile(r"uid0*(\d+)") 
uid_pattern

In [ ]:
for uid_dir in tacticas_root.iterdir():
    if not uid_dir.is_dir():
        continue
    
    #Try to extract sid from folder name
    match = uid_pattern.match(uid_dir.name)
    if not match:
        print(f"Skipping invalid folder name: {uid_dir.name}")
        continue

    sid = int(match.group(1))
    
    if sid not in sid_to_files:
        print(f"Skipping sid {sid}: not in df8918")
        continue

    allowed_files = sid_to_files[sid]
    print(f"Processing sid {sid} with {len(allowed_files)} files...")

    # Create output directory
    output_dir = output_root / uid_dir.name
    output_dir.mkdir(parents=True, exist_ok=True)

    # Iterate through all .wav files in the folder named uid
    for wav_file in uid_dir.glob("*.wav"):
        filename = wav_file.name
        if filename in allowed_files:
            output_path = output_dir / filename
            print(f"  Converting {wav_file} → {output_path}")
            try:
                tts.voice_conversion_to_file(
                    source_wav=str(wav_file),
                    target_wav=target_wav,
                    file_path=str(output_path)
                )
            except Exception as e:
                print(f" Failed to convert {wav_file}: {e}")
        else:
            print(f"  Skipping {filename}: not in df8918 for sid {sid}")

print(" All done!")

In [ ]:
#Verify that the SIDs in df8918 are all in the converted folder
converted_sids = set()  
for uid_dir in output_root.iterdir():
    if uid_dir.is_dir():
        match = uid_pattern.match(uid_dir.name)
        if match:
            sid = int(match.group(1))
            converted_sids.add(sid)
            